In [125]:
import pandas as pd
import numpy as np

### Load tournament results

In [126]:
tourney_men_df = pd.read_csv("../data/MNCAATourneyCompactResults.csv")
tourney_women_df = pd.read_csv("../data/WNCAATourneyCompactResults.csv")

- Because the season data from `_RegularSeasonDetailedResults.csv` goes from 2003 -> 2026 for men and 2010 -> 2025 for women, we need to filter out the dataframe to correspond to these years

In [127]:
tourney_men_df = tourney_men_df[tourney_men_df["Season"] >= 2003].reset_index(drop=True)
tourney_women_df = tourney_women_df[tourney_women_df["Season"] >= 2010].reset_index(drop=True)

In [128]:
tourney_men_df.head(5)

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
0,2003,134,1421,92,1411,84,N,1
1,2003,136,1112,80,1436,51,N,0
2,2003,136,1113,84,1272,71,N,0
3,2003,136,1141,79,1166,73,N,0
4,2003,136,1143,76,1301,74,N,1


### Build historical rows

In [129]:
men_hist = tourney_men_df.copy()
men_hist_rev = tourney_men_df.copy()

Now we need to convert each tournament game into 2 rows, where first row has the winner with a target of 1(winner) and the other team with the target of 0 (loser)

### Team1 wins

In [130]:
men_hist["Team1ID"] = men_hist["WTeamID"]
men_hist["Team2ID"] = men_hist["LTeamID"]
men_hist["Target"] = 1

### Team1 loses

In [131]:
men_hist_rev["Team1ID"] = men_hist_rev["LTeamID"]
men_hist_rev["Team2ID"] = men_hist_rev["WTeamID"]
men_hist_rev["Target"] = 0

Merge both datasets together

In [132]:
men_hist = pd.concat([men_hist, men_hist_rev], ignore_index=True)

In [133]:
men_hist = men_hist[["Season", "DayNum", "WLoc", "NumOT", "Team1ID", "Team2ID", "Target"]]
men_hist.head()

,Season,DayNum,WLoc,NumOT,Team1ID,Team2ID,Target
0,2003,134,N,1,1421,1411,1
1,2003,136,N,0,1112,1436,1
2,2003,136,N,0,1113,1272,1
3,2003,136,N,0,1141,1166,1
4,2003,136,N,1,1143,1301,1


### Now we do the same thing for women

In [134]:
women_hist = tourney_women_df.copy()
women_hist_rev = women_hist.copy()

In [135]:
women_hist["Team1ID"] = women_hist["WTeamID"]
women_hist["Team2ID"] = women_hist["LTeamID"]
women_hist["Target"] = 1

In [136]:
women_hist_rev["Team1ID"] = women_hist_rev["LTeamID"]
women_hist_rev["Team2ID"] = women_hist_rev["WTeamID"]
women_hist_rev["Target"] = 0

In [137]:
women_hist = pd.concat([women_hist, women_hist_rev], ignore_index=True)
women_hist = women_hist[["Season", "DayNum", "WLoc", "NumOT", "Team1ID", "Team2ID", "Target"]]

In [138]:
women_hist.head()

,Season,DayNum,WLoc,NumOT,Team1ID,Team2ID,Target
0,2010,138,N,0,3124,3201,1
1,2010,138,N,0,3173,3395,1
2,2010,138,H,0,3181,3214,1
3,2010,138,H,0,3199,3256,1
4,2010,138,N,0,3207,3265,1


### Merge with team_features

In [139]:
team_features_men = pd.read_csv("../data/m_team_season_features.csv")
team_features_women = pd.read_csv("../data/w_team_season_features.csv")

By separating the teams by winners and losers, it allows us to calculate all the different features between each team

In [140]:
team1_features = team_features_men.add_prefix("Team1_")
team2_features = team_features_men.add_prefix("Team2_")

In [141]:
men_matchups = men_hist.merge(
    team1_features,
    left_on=["Season", "Team1ID"],
    right_on=["Team1_Season", "Team1_TeamID"],
    how="left"
)

In [142]:
men_matchups = men_matchups.merge(
    team2_features,
    left_on=["Season", "Team2ID"],
    right_on=["Team2_Season", "Team2_TeamID"],
    how="left"
)

In [143]:
men_matchups.head()

,Season,DayNum,WLoc,NumOT,Team1ID,Team2ID,Target,Team1_Season,Team1_TeamID,Team1_Wins,...,Team2_AvgNetRating,Team2_AvgGameTotalPoints,Team2_Losses,Team2_WinPct,Team2_LossPct,Team2_ConfAbbrev,Team2_Seed,Team2_SeedNum,Team2_HasTournamentSeed,Team2_MasseyOrdinalRank
0,2003,134,N,1,1421,1411,1,2003,1421,13,...,0.020735,143.633333,12,0.600000,0.400000,swac,X16a,16.0,1,249.0
1,2003,136,N,0,1112,1436,1,2003,1112,25,...,0.074483,130.931034,10,0.655172,0.344828,aec,Z16,16.0,1,148.0
2,2003,136,N,0,1113,1272,1,2003,1113,18,...,0.124869,140.344828,6,0.793103,0.206897,cusa,Z07,7.0,1,18.0
3,2003,136,N,0,1141,1166,1,2003,1141,23,...,0.211501,143.575758,4,0.878788,0.121212,mvc,Z06,6.0,1,19.0
4,2003,136,N,1,1143,1301,1,2003,1143,21,...,0.064883,140.400000,12,0.600000,0.400000,acc,W09,9.0,1,48.0


### Feature engineering
- WinPctDiff
- WinPctGap
- SeedNumDiff
- SeedGap
- NetRatingDiff
- NetRatingGap

In [145]:
men_matchups["WinPctDiff"] = men_matchups["Team1_WinPct"] - men_matchups["Team2_WinPct"]
men_matchups["WinPctGap"] = np.abs(men_matchups["WinPctDiff"])

In [146]:
men_matchups["SeedNumDiff"] = men_matchups["Team1_SeedNum"] - men_matchups["Team2_SeedNum"]
men_matchups["SeedGap"] = np.abs(men_matchups["SeedNumDiff"])

In [147]:
men_matchups["NetRatingDiff"] = men_matchups["Team1_AvgNetRating"] - men_matchups["Team2_AvgNetRating"]
men_matchups["NetRatingGap"] = np.abs(men_matchups["NetRatingDiff"])

In [148]:
men_matchups["OffEffDiff"] = (
    men_matchups["Team1_AvgOffEfficiency"]
    - men_matchups["Team2_AvgOffEfficiency"]
)


In [149]:
men_matchups["DefEffDiff"] = (
    men_matchups["Team1_AvgDefEfficiency"]
    - men_matchups["Team2_AvgDefEfficiency"]
)

C:\Users\Barderus_Legion\AppData\Local\Temp\ipykernel_23592\3336336285.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  men_matchups["DefEffDiff"] = (


In [150]:
men_matchups["MarginDiff"] = (
    men_matchups["Team1_AvgMarginScore"]
    - men_matchups["Team2_AvgMarginScore"]
)

C:\Users\Barderus_Legion\AppData\Local\Temp\ipykernel_23592\3751298322.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  men_matchups["MarginDiff"] = (


In [151]:
men_matchups["ReboundPctDiff"] = (
    men_matchups["Team1_AvgReboundPct"]
    - men_matchups["Team2_AvgReboundPct"]
)

C:\Users\Barderus_Legion\AppData\Local\Temp\ipykernel_23592\3151197003.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  men_matchups["ReboundPctDiff"] = (


In [152]:
men_matchups["TurnoverPctDiff"] = (
    men_matchups["Team1_AvgTurnoverPct"]
    - men_matchups["Team2_AvgTurnoverPct"]
)

C:\Users\Barderus_Legion\AppData\Local\Temp\ipykernel_23592\4218104749.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  men_matchups["TurnoverPctDiff"] = (


In [153]:
men_matchups["FGPctDiff"] = (
    men_matchups["Team1_AvgFieldGoalsPct"]
    - men_matchups["Team2_AvgFieldGoalsPct"]
)

C:\Users\Barderus_Legion\AppData\Local\Temp\ipykernel_23592\2620259579.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  men_matchups["FGPctDiff"] = (


In [154]:
men_matchups["ThreePctDiff"] = (
    men_matchups["Team1_AvgThreePointsPct"]
    - men_matchups["Team2_AvgThreePointsPct"]
)

C:\Users\Barderus_Legion\AppData\Local\Temp\ipykernel_23592\418856106.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  men_matchups["ThreePctDiff"] = (


In [155]:
men_matchups["FTPctDiff"] = (
    men_matchups["Team1_AvgFreeThrowPct"]
    - men_matchups["Team2_AvgFreeThrowPct"]
)

C:\Users\Barderus_Legion\AppData\Local\Temp\ipykernel_23592\4053364020.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  men_matchups["FTPctDiff"] = (


In [156]:
men_matchups["MasseyRankDiff"] = (
    men_matchups["Team1_MasseyOrdinalRank"]
    - men_matchups["Team2_MasseyOrdinalRank"]
)

C:\Users\Barderus_Legion\AppData\Local\Temp\ipykernel_23592\2858725874.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  men_matchups["MasseyRankDiff"] = (


In [157]:
men_matchups = men_matchups.copy()

### Build dataset for training

For training we need:
 - Team identifiers
  - seasons
  - target variable
  - engineered features

Right now we only have 12 features, but we can play around and see how we can add or remove them as we go. I believe we have a total of 100(?) features to use =)

In [158]:
training_df = men_matchups[
    [
        "Season",
        "Team1ID",
        "Team2ID",
        "Target",

        # Core features
        "WinPctDiff",
        "SeedNumDiff",
        "NetRatingDiff",

        # Advanced
        "OffEffDiff",
        "DefEffDiff",
        "MarginDiff",
        "ReboundPctDiff",
        "TurnoverPctDiff",
        "FGPctDiff",
        "ThreePctDiff",
        "FTPctDiff",

        # Ratings
        "MasseyRankDiff"
    ]
].copy()

training_df.head()

,Season,Team1ID,Team2ID,Target,WinPctDiff,SeedNumDiff,NetRatingDiff,OffEffDiff,DefEffDiff,MarginDiff,ReboundPctDiff,TurnoverPctDiff,FGPctDiff,ThreePctDiff,FTPctDiff,MasseyRankDiff
0,2003,1421,1411,1,-0.151724,0.0,-0.118699,-0.021731,0.096968,-9.208046,-0.030010,0.012414,-0.016123,0.042080,0.152397,16.0
1,2003,1112,1436,1,0.237685,-15.0,0.120916,0.081979,-0.038936,10.309113,-0.013179,-0.019188,0.017191,-0.006859,0.051446,-145.0
2,2003,1113,1272,1,-0.172414,3.0,-0.023829,0.040148,0.063977,-1.896552,0.010731,0.006264,0.042222,-0.015065,0.047369,22.0
3,2003,1141,1166,1,-0.085684,5.0,-0.126746,-0.040490,0.086256,-8.805643,0.007251,0.055825,0.008041,-0.007432,0.073034,17.0
4,2003,1143,1301,1,0.124138,-1.0,0.002024,-0.020811,-0.022835,0.324138,0.009767,-0.012174,0.010234,0.025371,-0.089516,-18.0


In [160]:
training_df["MasseyRankDiff"] = training_df["MasseyRankDiff"].fillna(0)

In [159]:
training_df.isna().sum().sort_values(ascending=False)

MasseyRankDiff     512
Season               0
Team2ID              0
Team1ID              0
WinPctDiff           0
SeedNumDiff          0
NetRatingDiff        0
Target               0
OffEffDiff           0
DefEffDiff           0
ReboundPctDiff       0
MarginDiff           0
TurnoverPctDiff      0
FGPctDiff            0
ThreePctDiff         0
FTPctDiff            0
dtype: int64

In [161]:
training_df["Target"].value_counts()

Target
1    1449
0    1449
Name: count, dtype: int64

In [166]:
missing_rows = men_matchups[men_matchups["Team1_WinPct"].isna()]
missing_rows

,Season,DayNum,WLoc,NumOT,Team1ID,Team2ID,Target,Team1_Season,Team1_TeamID,Team1_Wins,...,NetRatingGap,OffEffDiff,DefEffDiff,MarginDiff,ReboundPctDiff,TurnoverPctDiff,FGPctDiff,ThreePctDiff,FTPctDiff,MasseyRankDiff


In [167]:
print("Training dataset shape:", training_df.shape)
print("Missing values:", training_df.isna().sum().sum())
print("Target distribution:")
print(training_df["Target"].value_counts(normalize=True))

Training dataset shape: (2898, 16)
Missing values: 0
Target distribution:
Target
1    0.5
0    0.5
Name: proportion, dtype: float64


In [ ]:
training_df.to_csv("../data/m_tournament_training_dataset.csv", index=False)